In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

In [5]:
class RiskEngine:
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100)
        self.encoders = {}
        self.features = ['carrier', 'weather', 'warehouse_load_pct', 'traffic_density']

    def train(self, data_path="historical_logistics.csv"):
        df = pd.read_csv(data_path)
        
        # Encoding categorical strings to numbers
        for col in ['carrier', 'weather']:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            self.encoders[col] = le
            
        X = df[self.features]
        y = df['is_delayed']
        
        self.model.fit(X, y)
        print("Risk Engine trained on historical patterns.")

    def predict_likelihood(self, live_data):
        """
        Takes a dictionary of a single shipment's data.
        Returns a float between 0 and 1.
        """
        # Prepare the data (must match training format)
        df_input = pd.DataFrame([live_data])
        for col in ['carrier', 'weather']:
            df_input[col] = self.encoders[col].transform(df_input[col])
            
        # Get probability of class 1 (Delayed)
        probs = self.model.predict_proba(df_input[self.features])
        return probs[0][1]